# Day 18: Advanced Qdrant Operations - Filtered Searches & Scroll APIs

Welcome to Day 18! Today we tackle a critical component of real-world AI engineering: **Managing and querying large vector datasets.**

When building production RAG systems or semantic search engines, simply returning the top K nearest neighbors based purely on vector similarity is rarely sufficient. Real applications require:
1.  **Metadata Filtering:** Filtering out documents based on structured metadata (e.g., date, author, permissions, category) before or during the similarity search. This ensures users only see data they are authorized to see or that is contextually relevant.
2.  **Pagination and Scrolling:** When you have thousands of matches or need to iterate over large datasets for batch processing (like migrations or offline analytics), you need a reliable way to cursor through data without loading everything into memory.

## The "Why" and "How"

**Filtering (Payload Filtering):**
Qdrant allows you to attach JSON payloads (metadata) to your vectors. Filtered searches apply conditions to these payloads. Qdrant handles this efficiently at the engine level, combining dense vector indices (like HNSW) with payload indices. This is crucial for *multi-tenant* applications where data separation is required.

**Scroll API:**
The Scroll API is designed to sequentially iterate over all (or filtered) records in a collection. It uses an `offset` (a point ID) to fetch the next batch. This is fundamentally different from a vector search; it's a structural traversal. It's the standard pattern for data dumps, re-indexing, and background batch jobs.

Let's dive into the production-grade implementation of these operations.

## AI Security Implications (PII & Fallbacks)
When implementing filtered searches and batch processing with tools like Qdrant, we must adhere to key AI security principles:
* **PII Protection:** Never embed raw Personally Identifiable Information (PII) into dense vectors or store them plainly in unencrypted payloads. Ensure data filtering occurs to segregate PII and use robust access controls before indexing.
* **Fallbacks:** In production, network calls to vector databases can fail. We must implement fallback mechanisms (like try-except blocks wrapping API calls) returning graceful degradation responses rather than crashing the system or exposing internal error logs.
* **Prompt Injection / Data Poisoning:** If vector payloads or database searches are user-driven, sanitize inputs meticulously to prevent malicious filter strings from bypassing authorization logic.

## 1. Setup & Basic Tier: Core Filtering
First, we'll initialize an in-memory Qdrant client, ingest dummy records, and perform a basic filtered search. This isolates the core concept with minimal boilerplate.

In [1]:
from typing import List, Dict, Any
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, Filter, FieldCondition, MatchValue
import uuid

# Basic Setup & Core Concept
def basic_filtered_search():
    client = QdrantClient(":memory:")
    collection = "basic_tech"
    
    # Initialize collection
    client.create_collection(
        collection_name=collection,
        vectors_config=VectorParams(size=3, distance=Distance.COSINE),
    )
    
    # Ingest minimal data
    client.upsert(
        collection_name=collection,
        points=[
            PointStruct(id=1, vector=[0.1, 0.2, 0.3], payload={"author": "Alice", "secret": False}),
            PointStruct(id=2, vector=[0.9, 0.8, 0.7], payload={"author": "Bob", "secret": True})
        ]
    )
    
    # Basic Filtered Search
    query_filter = Filter(must=[FieldCondition(key="author", match=MatchValue(value="Alice"))])
    results = client.query_points(
        collection_name=collection,
        query=[0.1, 0.2, 0.3],
        query_filter=query_filter,
        limit=1
    )
    
    print("--- Basic Tier Results ---")
    for hit in results.points:
         print(f"ID: {hit.id}, Score: {hit.score:.4f}, Payload: {hit.payload}")

basic_filtered_search()


--- Basic Tier Results ---
ID: 1, Score: 1.0000, Payload: {'author': 'Alice', 'secret': False}


## 2. Medium Tier: Clean OOP & State Management
This tier wraps the operations in a class, demonstrating state management, clean OOP principles, and multi-tenant search handling.

In [2]:
class TenantVectorStore:
    def __init__(self, collection_name: str):
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name
        self._initialize()

    def _initialize(self):
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=4, distance=Distance.COSINE)
        )

    def add_documents(self, points: List[PointStruct]):
        self.client.upsert(
            collection_name=self.collection_name,
            points=points
        )

    def search_by_tenant(self, tenant_id: str, query_vector: List[float]) -> None:
        tenant_filter = Filter(must=[FieldCondition(key="tenant_id", match=MatchValue(value=tenant_id))])
        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_vector,
            query_filter=tenant_filter,
            limit=5
        )
        print(f"--- Medium Tier: Tenant {tenant_id} Results ---")
        for hit in results.points:
            print(f"ID: {hit.id}, Payload: {hit.payload}")

# Execution
store = TenantVectorStore("tenant_data")
store.add_documents([
    PointStruct(id=1, vector=[0.1, 0.2, 0.3, 0.4], payload={"tenant_id": "T1", "content": "Hello T1"}),
    PointStruct(id=2, vector=[0.5, 0.5, 0.5, 0.5], payload={"tenant_id": "T2", "content": "Hello T2"}),
])
store.search_by_tenant("T1", [0.1, 0.2, 0.3, 0.4])


--- Medium Tier: Tenant T1 Results ---
ID: 1, Payload: {'tenant_id': 'T1', 'content': 'Hello T1'}


## 3. Advanced Tier: Production-Grade Implementation & Scroll API
This section implements a robust, type-hinted manager with error handling (fallbacks), exact imports, docstrings, and safe Scroll API batching. It embodies IDE-less interview readiness.

In [3]:
import logging
from typing import Optional, List, Tuple
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, Filter, FieldCondition, MatchValue, Record
from qdrant_client.http.exceptions import UnexpectedResponse

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ProductionBatchProcessor:
    """
    A production-grade processor for safely iterating over vector collections using the Scroll API.
    Demonstrates clean OOP, strict type hinting, and robust error handling (fallbacks).
    """
    def __init__(self, client: QdrantClient, collection_name: str):
        self.client = client
        self.collection_name = collection_name
        self._ensure_collection()

    def _ensure_collection(self) -> None:
        try:
            if not self.client.collection_exists(self.collection_name):
                 self.client.create_collection(
                     collection_name=self.collection_name,
                     vectors_config=VectorParams(size=4, distance=Distance.COSINE)
                 )
        except Exception as e:
            logger.error(f"Failed to initialize collection: {e}")
            raise

    def process_all_records(self, batch_size: int = 2) -> None:
        """
        Iterates over all records in the collection using the Scroll API.
        Implements a fallback mechanism if the API call fails.
        """
        next_page_offset: Optional[int] = None
        
        while True:
            try:
                records, next_page_offset = self.client.scroll(
                    collection_name=self.collection_name,
                    limit=batch_size,
                    offset=next_page_offset,
                    with_payload=True,
                    with_vectors=False
                )
                
                for record in records:
                    # Simulating processing
                    pass
                logger.info(f"Successfully processed batch of {len(records)} records.")
                
                if next_page_offset is None:
                    logger.info("Reached end of collection.")
                    break
                    
            except UnexpectedResponse as ur:
                logger.error(f"Qdrant API Error during scroll: {ur}. Triggering fallback.")
                break # Fallback: exit cleanly instead of crashing
            except Exception as e:
                logger.error(f"Unexpected system error: {e}. Triggering fallback.")
                break

# Execution
adv_client = QdrantClient(":memory:")
processor = ProductionBatchProcessor(adv_client, "prod_data")
# Ingest some dummy data
adv_client.upsert(
    collection_name="prod_data", 
    points=[
        PointStruct(id=1, vector=[0.1]*4, payload={"status": "active"}),
        PointStruct(id=2, vector=[0.2]*4, payload={"status": "inactive"}),
        PointStruct(id=3, vector=[0.3]*4, payload={"status": "active"})
    ]
)
processor.process_all_records(batch_size=2)


INFO:__main__:Successfully processed batch of 2 records.


INFO:__main__:Successfully processed batch of 1 records.


INFO:__main__:Reached end of collection.


## Common Pitfalls in Production

1.  **Missing Payload Indices:** Filtering on unindexed payload fields requires a full scan of the dataset, which is disastrous for performance. *Always* create payload indices (`client.create_payload_index(...)`) for fields you frequently filter by.
2.  **Using Search for Batch Export:** Developers sometimes try to use `client.search` with a massive `limit` to export data. This overloads the memory and the HNSW index. Use `client.scroll` instead.
3.  **Complex Filter Logic:** Deeply nested `must`, `should`, and `must_not` clauses can become difficult to debug. Keep filter logic as flat as possible, or handle complex business logic upstream before querying the database.
4.  **Offset Mismanagement:** When implementing pagination on the frontend, relying on the `offset` parameter in `search` for deep pagination is inefficient. It forces the engine to calculate all results up to `offset + limit`. For deep pagination or full traversal, use the Scroll API.

### Reference Links
* [Qdrant Documentation: Filtering](https://qdrant.tech/documentation/concepts/filtering/)
* [Qdrant Documentation: Pagination & Scroll API](https://qdrant.tech/documentation/concepts/explore/#scroll-api)
* [LangChain Vector Store Integrations](https://python.langchain.com/v0.2/docs/integrations/vectorstores/qdrant/)

## Practical Lab: Multi-Tenant Batch Processor

**Your Task:**
You are building an administrative tool for a multi-tenant platform.

1.  Create a function `process_tenant_data(client, collection_name, author_name)` that uses the **Scroll API** to retrieve *all* records authored by a specific `author_name`.
2.  The scroll should fetch records in batches of 2.
3.  Print the total number of records found for that author.

*Hint: You will need to combine the `scroll` method with a `Filter` object, similar to how we used it in the `search` method.*

**Video Walkthrough Requirement:**
Once you complete this code implementation, please record a brief async video walkthrough (e.g., using Loom) explaining your design decisions, specifically focusing on how you incorporated OOP principles and AI security measures.

In [4]:
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance, Filter, FieldCondition, MatchValue

# Lab Implementation Setup
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="tech_articles",
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)
client.upsert(
    collection_name="tech_articles",
    points=[
        PointStruct(id=1, vector=[0.1]*4, payload={"author": "Alice", "title": "A"}),
        PointStruct(id=2, vector=[0.2]*4, payload={"author": "Bob", "title": "B"}),
        PointStruct(id=3, vector=[0.3]*4, payload={"author": "Alice", "title": "C"}),
    ]
)

def process_tenant_data(client: QdrantClient, collection_name: str, author_name: str, batch_size: int = 2) -> None:
    """
    Uses the Scroll API to retrieve all records for a specific author in batches.
    """
    # Define the filter for the specific author
    author_filter = Filter(
        must=[
            FieldCondition(
                key="author",
                match=MatchValue(value=author_name)
            )
        ]
    )
    
    print(f"\n--- Processing Data for Tenant/Author: {author_name} ---")
    next_page_offset = None
    total_records = 0
    iteration = 1
    
    while True:
        records, next_page_offset = client.scroll(
            collection_name=collection_name,
            scroll_filter=author_filter,
            limit=batch_size,
            offset=next_page_offset,
            with_payload=True,
            with_vectors=False
        )
        
        batch_count = len(records)
        total_records += batch_count
        print(f"Batch {iteration}: Retrieved {batch_count} records.")
        for record in records:
             print(f"  Record ID: {record.id}, Payload: {record.payload}")
             
        if next_page_offset is None:
            break
            
        iteration += 1
        
    print(f"Total records processed for {author_name}: {total_records}")

# Execute Lab task for author "Alice"
process_tenant_data(client, "tech_articles", "Alice")



--- Processing Data for Tenant/Author: Alice ---
Batch 1: Retrieved 2 records.
  Record ID: 1, Payload: {'author': 'Alice', 'title': 'A'}
  Record ID: 3, Payload: {'author': 'Alice', 'title': 'C'}
Total records processed for Alice: 2
